# Predicting Residential Property Values in Calgary

This project predicts residential property assessed values in Calgary using data from the [City of Calgary Open Data Portal](https://data.calgary.ca). Two models are compared: OLS regression (baseline) and Random Forest. The Random Forest model achieves an R² of 0.86, significantly outperforming OLS (R² = 0.61).

**Dataset:** ~490,000 residential properties from the 2026 assessment roll  
**Features:** Lot size, property age, dwelling type, land use designation, and community-level attributes  
**Tools:** Python, pandas, scikit-learn, matplotlib, seaborn

## 1. Data Loading & Initial Exploration

In [ ]:
import pandas as pd

df = pd.read_csv('data/Current_Year_Property_Assessments.csv')

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head(3))

In [ ]:
print(df['ASSESSMENT_CLASS'].value_counts())
print(df['PROPERTY_TYPE'].value_counts())

## 2. Data Cleaning

Filter to residential properties with structures. Remove commercial, farmland, vacant lots, and administrative records. Fix string-formatted numeric columns.

In [ ]:
df_res = df[df['ASSESSMENT_CLASS'] == 'RE'].copy()
print(df_res['SUB_PROPERTY_USE'].value_counts().head(15))

In [ ]:
# Check average assessed values by sub-property type to confirm categories make sense
df_res['ASSESSED_VALUE_NUM'] = df_res['ASSESSED_VALUE'].str.replace(',', '').astype(float)

print(df_res.groupby('SUB_PROPERTY_USE')['ASSESSED_VALUE_NUM'].median().sort_values(ascending=False).head(15))

In [ ]:
# Median values for the most common sub-property types only
common_types = ['R110', 'R120', 'R201', 'R202', 'R301', 'R302', 'R401', 'R402', 'A003', 'A004', 'A005', 'A006']
print(df_res[df_res['SUB_PROPERTY_USE'].isin(common_types)].groupby('SUB_PROPERTY_USE')['ASSESSED_VALUE_NUM'].median())

In [ ]:
# Make a proper copy and filter to residential R-type properties
df_res = df[(df['ASSESSMENT_CLASS'] == 'RE') & (df['SUB_PROPERTY_USE'].str.startswith('R'))].copy()

# Fix string columns to numeric
for col in ['ASSESSED_VALUE', 'LAND_SIZE_SF', 'LAND_SIZE_SM', 'LAND_SIZE_AC']:
    df_res[col] = pd.to_numeric(df_res[col].str.replace(',', ''), errors='coerce')

# Create dwelling type groups
dwelling_map = {
    'R110': 'Detached', 'R111': 'Detached',
    'R120': 'Semi-Detached', 'R121': 'Semi-Detached',
    'R201': 'Townhouse', 'R202': 'Townhouse',
    'R301': 'Low-Rise Condo', 'R302': 'Low-Rise Condo',
    'R401': 'High-Rise Condo', 'R402': 'High-Rise Condo',
    'R510': 'Manufactured'
}
df_res['DWELLING_TYPE'] = df_res['SUB_PROPERTY_USE'].map(dwelling_map).fillna('Other')

# Create property age
df_res['PROPERTY_AGE'] = 2026 - df_res['YEAR_OF_CONSTRUCTION']

# Check result
print(df_res.shape)
print(df_res['DWELLING_TYPE'].value_counts())
print(df_res[['ASSESSED_VALUE', 'LAND_SIZE_SF', 'PROPERTY_AGE']].describe())

### Outlier Removal

Remove properties with $0 assessed values and trim the top/bottom 1% to prevent extreme outliers from distorting the models.

In [ ]:
# Remove zeros and extreme outliers
print(f"Before cleaning: {df_res.shape[0]}")

# Drop $0 assessed values
df_res = df_res[df_res['ASSESSED_VALUE'] > 0]

# Drop extreme outliers (top/bottom 1% of assessed value)
q01 = df_res['ASSESSED_VALUE'].quantile(0.01)
q99 = df_res['ASSESSED_VALUE'].quantile(0.99)
df_res = df_res[(df_res['ASSESSED_VALUE'] >= q01) & (df_res['ASSESSED_VALUE'] <= q99)]

# Drop missing values in key columns
df_res = df_res.dropna(subset=['LAND_SIZE_SF', 'YEAR_OF_CONSTRUCTION'])

# Drop "Other" and "Manufactured" dwelling types (too few observations)
df_res = df_res[~df_res['DWELLING_TYPE'].isin(['Other', 'Manufactured'])]

print(f"After cleaning: {df_res.shape[0]}")
print(df_res[['ASSESSED_VALUE', 'LAND_SIZE_SF', 'PROPERTY_AGE']].describe())

### Community-Level Feature Engineering

Create neighborhood-level variables: median assessed value, median property age, percentage of detached homes, and property count per community. Drop communities with fewer than 30 properties.

In [ ]:
# Compute community-level features
comm_stats = df_res.groupby('COMM_NAME').agg(
    COMM_MEDIAN_VALUE=('ASSESSED_VALUE', 'median'),
    COMM_MEDIAN_AGE=('PROPERTY_AGE', 'median'),
    COMM_PCT_DETACHED=('DWELLING_TYPE', lambda x: (x == 'Detached').mean()),
    COMM_PROPERTY_COUNT=('ASSESSED_VALUE', 'count')
).reset_index()

# Merge back to main dataframe
df_res = df_res.merge(comm_stats, on='COMM_NAME', how='left')

# Drop tiny communities (fewer than 30 properties — unreliable stats)
print(f"Before dropping small communities: {df_res.shape[0]}")
df_res = df_res[df_res['COMM_PROPERTY_COUNT'] >= 30]
print(f"After: {df_res.shape[0]}")

print(comm_stats.describe())

## 3. Feature Selection

In [ ]:
# Final feature set
features = ['LAND_SIZE_SF', 'PROPERTY_AGE', 'DWELLING_TYPE', 'LAND_USE_DESIGNATION',
            'COMM_MEDIAN_VALUE', 'COMM_MEDIAN_AGE', 'COMM_PCT_DETACHED']
target = 'ASSESSED_VALUE'

# Check what we're working with
print(f"Final dataset: {df_res.shape[0]} observations")
print(f"Target: {target}")
print(f"Features: {len(features)}")
print(f"\nNumeric features:")
print(df_res[['ASSESSED_VALUE', 'LAND_SIZE_SF', 'PROPERTY_AGE', 
              'COMM_MEDIAN_VALUE', 'COMM_MEDIAN_AGE', 'COMM_PCT_DETACHED']].describe().round(2))
print(f"\nCategorical features:")
print(df_res['DWELLING_TYPE'].value_counts())
print(f"\nLand use designations (top 10):")
print(df_res['LAND_USE_DESIGNATION'].value_counts().head(10))

In [ ]:
# Group rare land use designations into 'Other'
top_land_use = df_res['LAND_USE_DESIGNATION'].value_counts().head(10).index
df_res['LAND_USE_GROUP'] = df_res['LAND_USE_DESIGNATION'].where(
    df_res['LAND_USE_DESIGNATION'].isin(top_land_use), 'Other'
)

print(df_res['LAND_USE_GROUP'].value_counts())

## 4. Exploratory Data Analysis

Visualize distributions, relationships between features and assessed values, and check for multicollinearity.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of assessed values
axes[0, 0].hist(df_res['ASSESSED_VALUE'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Assessed Values')
axes[0, 0].set_xlabel('Assessed Value ($)')
axes[0, 0].set_ylabel('Frequency')

# 2. Assessed value by dwelling type
df_res.boxplot(column='ASSESSED_VALUE', by='DWELLING_TYPE', ax=axes[0, 1], vert=True)
axes[0, 1].set_title('Assessed Value by Dwelling Type')
axes[0, 1].set_xlabel('Dwelling Type')
axes[0, 1].set_ylabel('Assessed Value ($)')
axes[0, 1].tick_params(axis='x', rotation=45)
plt.sca(axes[0, 1])
plt.title('Assessed Value by Dwelling Type')

# 3. Property age vs assessed value (scatter — sample to avoid overplotting)
sample = df_res.sample(5000, random_state=42)
axes[1, 0].scatter(sample['PROPERTY_AGE'], sample['ASSESSED_VALUE'], alpha=0.3, s=5)
axes[1, 0].set_title('Property Age vs Assessed Value')
axes[1, 0].set_xlabel('Property Age (years)')
axes[1, 0].set_ylabel('Assessed Value ($)')

# 4. Lot size vs assessed value (log scale helps with skew)
axes[1, 1].scatter(sample['LAND_SIZE_SF'], sample['ASSESSED_VALUE'], alpha=0.3, s=5)
axes[1, 1].set_title('Lot Size vs Assessed Value')
axes[1, 1].set_xlabel('Lot Size (sq ft)')
axes[1, 1].set_ylabel('Assessed Value ($)')
axes[1, 1].set_xscale('log')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns

numeric_cols = ['ASSESSED_VALUE', 'LAND_SIZE_SF', 'PROPERTY_AGE', 
                'COMM_MEDIAN_VALUE', 'COMM_MEDIAN_AGE', 'COMM_PCT_DETACHED']

corr = df_res[numeric_cols].corr().round(2)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Numeric Variables')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

Community median age and property age are highly correlated (r = 0.76). Drop community median age to avoid multicollinearity in the OLS model.

In [ ]:
# Drop COMM_MEDIAN_AGE due to multicollinearity with PROPERTY_AGE (r=0.76)
features = ['LAND_SIZE_SF', 'PROPERTY_AGE', 'DWELLING_TYPE', 'LAND_USE_GROUP',
            'COMM_MEDIAN_VALUE', 'COMM_PCT_DETACHED']

print(f"Final feature count: {len(features)}")
print(f"Final observations: {df_res.shape[0]}")

## 5. Model Training & Evaluation

Split data 80/20 into training and test sets. Train OLS regression as a baseline and Random Forest to capture non-linear relationships.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Prepare features — one-hot encode categoricals
df_model = pd.get_dummies(df_res[features + [target]], 
                          columns=['DWELLING_TYPE', 'LAND_USE_GROUP'], 
                          drop_first=True)

# Split
X = df_model.drop(columns=[target])
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} observations")
print(f"Test set: {X_test.shape[0]} observations")
print(f"Features after encoding: {X_train.shape[1]}")

### OLS Regression

In [ ]:
# OLS Regression
ols_model = LinearRegression()
ols_model.fit(X_train, y_train)

ols_pred = ols_model.predict(X_test)

ols_mse = mean_squared_error(y_test, ols_pred)
ols_rmse = np.sqrt(ols_mse)
ols_r2 = r2_score(y_test, ols_pred)

print("=== OLS Regression Results ===")
print(f"R-squared: {ols_r2:.4f}")
print(f"RMSE: ${ols_rmse:,.2f}")
print(f"\nCoefficients:")
coef_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': ols_model.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))
print(f"\nIntercept: ${ols_model.intercept_:,.2f}")

### Random Forest

In [ ]:
# Random Forest — use n_jobs=-1 to speed up
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, 
                                  min_samples_leaf=20, random_state=42, 
                                  n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("=== Random Forest Results ===")
print(f"R-squared: {rf_r2:.4f}")
print(f"RMSE: ${rf_rmse:,.2f}")

# Feature importance
importance_df = pd.DataFrame({'Feature': X_train.columns, 
                               'Importance': rf_model.feature_importances_})
importance_df = importance_df.sort_values('Importance', ascending=False)
print(f"\nFeature Importance:")
print(importance_df.to_string(index=False))

## 6. Robustness Check

Re-run both models with a 70/30 split to verify results are stable and not dependent on a particular random split.

In [ ]:
# Robustness check — 70/30 split
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.3, random_state=99)

ols2 = LinearRegression().fit(X_train2, y_train2)
rf2 = RandomForestRegressor(n_estimators=100, max_depth=15, 
                             min_samples_leaf=20, random_state=42, 
                             n_jobs=-1).fit(X_train2, y_train2)

print("=== Robustness Check: 70/30 Split ===")
print(f"OLS  R²: {r2_score(y_test2, ols2.predict(X_test2)):.4f}  RMSE: ${np.sqrt(mean_squared_error(y_test2, ols2.predict(X_test2))):,.2f}")
print(f"RF   R²: {r2_score(y_test2, rf2.predict(X_test2)):.4f}  RMSE: ${np.sqrt(mean_squared_error(y_test2, rf2.predict(X_test2))):,.2f}")

print("\n=== Summary Table ===")
print(f"{'Split':<12} {'Model':<8} {'R²':<10} {'RMSE':<15}")
print(f"{'80/20':<12} {'OLS':<8} {ols_r2:<10.4f} ${ols_rmse:<15,.2f}")
print(f"{'80/20':<12} {'RF':<8} {rf_r2:<10.4f} ${rf_rmse:<15,.2f}")
print(f"{'70/30':<12} {'OLS':<8} {r2_score(y_test2, ols2.predict(X_test2)):<10.4f} ${np.sqrt(mean_squared_error(y_test2, ols2.predict(X_test2))):<15,.2f}")
print(f"{'70/30':<12} {'RF':<8} {r2_score(y_test2, rf2.predict(X_test2)):<10.4f} ${np.sqrt(mean_squared_error(y_test2, rf2.predict(X_test2))):<15,.2f}")

## 7. Feature Importance

In [ ]:
# Feature importance plot (top 10 only for readability)
top_10 = importance_df.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_10['Feature'], top_10['Importance'], color='steelblue', edgecolor='black')
plt.xlabel('Importance')
plt.title('Random Forest — Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Findings

- **Random Forest (R² = 0.86) significantly outperforms OLS (R² = 0.61)**, confirming non-linear relationships in the data.
- **Lot size** is the most important predictor in Random Forest (40.5% importance) despite being nearly useless in OLS — the relationship is non-linear.
- **Community median value** is the second strongest predictor (27.2%), reflecting the importance of location.
- Results are stable across different train-test splits (80/20 and 70/30).

### Limitations
- No interior characteristics (square footage, bedrooms, renovations)
- Assessed values are municipal appraisals, not market sale prices
- Community-level features are derived from the same dataset